# CNN/DBP-компенсация на PyTorch

Компактная замена MXNet-ноутбуков для текущего API `ber-equalization-studio`. Обучение, файловое разделение train/val/test, нормировка, BER и сохранение результатов выполняются библиотекой.

Соответствие архитектур:
- `MXNet_Complex_FCNN_Conv_GPU_v1` → `complex_fcnn_kerr`;
- `MXNet_CNN_DBP_1channel_v1_SeqStat` → `complex_dbp_seqstat`;
- физические одно-канальные блоки из `MXNet_DCNN_DBP_Stage3` используются в `complex_dbp_seqstat`. Межканальные XPM-ветви Stage3 не включены, потому что текущий CSV содержит одну принятую комплексную последовательность.

In [ ]:
from pathlib import Path

from ber_equalization_studio import Studio

ROOT = Path.cwd()
while not (ROOT / 'ber-equalization-studio').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'test'
OUT_DIR = ROOT / 'notebook_runs'
studio = Studio(data_dirs=[DATA_DIR], out_dir=OUT_DIR, device='cuda')

## Формат данных

Библиотека сама находит `Symbols_1m_1ch_PR_*.csv`. Столбцы: `tx_re, tx_im, rx_re, rx_im`. Окна строятся отдельно внутри каждого файла, поэтому границы реализаций не смешиваются, а test-файлы не участвуют в нормировке или выборе checkpoint.

In [ ]:
studio.models().query("model in ['complex_fcnn_kerr', 'complex_dbp_seqstat']")

## Обучение

DBP-настройки ниже намеренно компактнее исходных 20 шагов × 151 taps: четыре обучаемых шага помещаются в `context_k=24` и подходят для первого рабочего эксперимента. После проверки можно увеличивать `dbp_num_steps`, размеры ядер и `context_k` совместно.

In [ ]:
run = studio.run(
    name='pytorch_cnn_dbp',
    models=['complex_fcnn_kerr', 'complex_dbp_seqstat'],
    context_k=24,
    epochs=50,
    lr=1e-3,
    max_test_files=1,
    fcnn_hidden_channels=32,
    dbp_num_steps=4,
    dbp_kernel_size=3,
    dbp_final_kernel_size=5,
    dbp_nl_memory=1,
)
run.results

In [ ]:
run.compare()

In [ ]:
run.plot_comparison()